# 1. Load logs

In [ ]:
import pandas as pd
from datetime import timedelta
import time

# LLM
from llm_helper_v2 import generate_incident_summary
from llm_helper_local import generate_incident_summary_local # LOCAL
from llm_helper_local_constrained_reasoning import generate_incident_summary_local_robust # More Robust

ImportError: cannot import name 'generate_incident_summary_local_robust' from 'llm_helper_local_constrained_reasoning' (c:\Users\eric-\Desktop\Projects\minisoc\llm_helper_local_constrained_reasoning.py)

In [2]:
hdfs = pd.read_csv("data/HDFS_2k.log_structured.csv")

In [3]:
auth = pd.read_csv("data/auth_logs.csv")
email = pd.read_csv("data/email_reports.csv")
endpoint = pd.read_csv("data/endpoint_logs.csv")

auth["timestamp"] = pd.to_datetime(auth["timestamp"])
email["timestamp"] = pd.to_datetime(email["timestamp"])
endpoint["timestamp"] = pd.to_datetime(endpoint["timestamp"])

In [4]:
hdfs.head()

,LineId,Date,Time,Pid,Level,Component,Content,EventId,EventTemplate
0,1,81109,203615,148,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_38865049064139...,E10,PacketResponder <*> for block blk_<*> terminating
1,2,81109,203807,222,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-6952295868487...,E10,PacketResponder <*> for block blk_<*> terminating
2,3,81109,204005,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...,E6,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,81109,204015,308,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_82291938032499...,E10,PacketResponder <*> for block blk_<*> terminating
4,5,81109,204106,329,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-6670958622368...,E10,PacketResponder <*> for block blk_<*> terminating


In [5]:
auth.head()

,timestamp,user,ip,country,action,status
0,2026-03-08 09:01:00,alice,10.0.0.12,FR,login,failed
1,2026-03-08 09:02:00,alice,10.0.0.12,FR,login,failed
2,2026-03-08 09:03:00,alice,10.0.0.12,FR,login,failed
3,2026-03-08 09:04:00,alice,185.220.101.4,RU,login,success
4,2026-03-08 10:15:00,bob,10.0.0.18,FR,vpn_login,success


In [6]:
email.head()

,ticket_id,timestamp,user,sender,subject,body,attachment,reported_reason
0,TCK-001,2026-03-08 08:58:00,alice,billing@micros0ft-support.com,Urgent invoice overdue,Please review the attached invoice and pay today,invoice.docm,suspicious attachment
1,TCK-002,2026-03-08 10:05:00,bob,hr@company.com,Policy update,Please read the latest policy PDF,policy.pdf,unknown sender
2,TCK-003,2026-03-08 11:05:00,charlie,security@okta-login-alert.com,Reset your password,Your password expires today click here,reset_link,no reason


In [7]:
endpoint.head()

,timestamp,hostname,user,process_name,file_name,action
0,2026-03-08 09:06:00,PC-ALICE,alice,WINWORD.EXE,invoice.docm,opened
1,2026-03-08 09:07:00,PC-ALICE,alice,powershell.exe,NaN,executed
2,2026-03-08 09:08:00,PC-ALICE,alice,cmd.exe,NaN,executed
3,2026-03-08 10:20:00,PC-BOB,bob,AcroRd32.exe,policy.pdf,opened
4,2026-03-08 11:07:00,PC-CHARLIE,charlie,chrome.exe,reset_link,opened


These datasets represent three typical sources of SOC data:
- authentication logs
- reported suspicious emails
- endpoint activity logs

# 2. Inspect dataset

In [8]:
auth.describe()

,timestamp
count,6
mean,2026-03-08 09:35:50
min,2026-03-08 09:01:00
25%,2026-03-08 09:02:15
50%,2026-03-08 09:03:30
75%,2026-03-08 09:57:15
max,2026-03-08 11:10:00


In [9]:
email.describe()

,timestamp
count,3
mean,2026-03-08 10:02:40
min,2026-03-08 08:58:00
25%,2026-03-08 09:31:30
50%,2026-03-08 10:05:00
75%,2026-03-08 10:35:00
max,2026-03-08 11:05:00


In [10]:
endpoint.describe()

,timestamp
count,5
mean,2026-03-08 09:45:36
min,2026-03-08 09:06:00
25%,2026-03-08 09:07:00
50%,2026-03-08 09:08:00
75%,2026-03-08 10:20:00
max,2026-03-08 11:07:00


In [11]:
auth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  6 non-null      datetime64[ns]
 1   user       6 non-null      object        
 2   ip         6 non-null      object        
 3   country    6 non-null      object        
 4   action     6 non-null      object        
 5   status     6 non-null      object        
dtypes: datetime64[ns](1), object(5)
memory usage: 420.0+ bytes


In [12]:
email.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ticket_id        3 non-null      object        
 1   timestamp        3 non-null      datetime64[ns]
 2   user             3 non-null      object        
 3   sender           3 non-null      object        
 4   subject          3 non-null      object        
 5   body             3 non-null      object        
 6   attachment       3 non-null      object        
 7   reported_reason  3 non-null      object        
dtypes: datetime64[ns](1), object(7)
memory usage: 324.0+ bytes


In [13]:
endpoint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   timestamp     5 non-null      datetime64[ns]
 1   hostname      5 non-null      object        
 2   user          5 non-null      object        
 3   process_name  5 non-null      object        
 4   file_name     3 non-null      object        
 5   action        5 non-null      object        
dtypes: datetime64[ns](1), object(5)
memory usage: 372.0+ bytes


# 3. Detect suspicious patterns

In [14]:
# Forget the password / Hack

failed_counts = auth[auth["status"] == "failed"].groupby("user").size()
failed_counts

user
alice    3
dtype: int64

In [15]:
# Use VPN

user_localisation = auth.groupby("user")["country"].nunique()
user_localisation[user_localisation >= 2]

# we can also add the time -> if the user switch location during the hours -> suspisious
# especially instant switch => VPN

user
alice    2
Name: country, dtype: int64

In [16]:
# Suspicious process

endpoint[endpoint["process_name"].str.contains("powershell", case=False, na=False)]

,timestamp,hostname,user,process_name,file_name,action
1,2026-03-08 09:07:00,PC-ALICE,alice,powershell.exe,NaN,executed


In [17]:
# Suspicious attachments -> SPAM

email[email["attachment"].str.contains(".docm", na=False)]

,ticket_id,timestamp,user,sender,subject,body,attachment,reported_reason
0,TCK-001,2026-03-08 08:58:00,alice,billing@micros0ft-support.com,Urgent invoice overdue,Please review the attached invoice and pay today,invoice.docm,suspicious attachment


In [18]:
# Check 
    # the email or name doesn't contains special characters or number (classical fraud)
    # if it is wrote by AI
    # Error or Vocabulary
    # If the attachement is joint with picture
    # Special Link
    # classical trap (.exe or somethings, need to check and other)

SOC analysts typically start with signals such as repeated login failures, suspicious attachments, or unusual process execution.
- Also check the object especially if there is a .docm. It is a macro that can run code

# 4. Correlate events

In [19]:
incidents = []

for _, row in email.iterrows():
    user = row["user"]
    start = row["timestamp"]
    end = start + timedelta(minutes=30)

    related_auth = auth[(auth["user"] == user) &
                        (auth["timestamp"] >= start) &
                        (auth["timestamp"] <= end)]

    related_endpoint = endpoint[(endpoint["user"] == user) &
                                (endpoint["timestamp"] >= start) &
                                (endpoint["timestamp"] <= end)]

    incidents.append({
        "user": user,
        "email_subject": row["subject"],
        "auth_events": len(related_auth),
        "endpoint_events": len(related_endpoint)
    })

incidents_df = pd.DataFrame(incidents)
incidents_df

,user,email_subject,auth_events,endpoint_events
0,alice,Urgent invoice overdue,4,3
1,bob,Policy update,1,1
2,charlie,Reset your password,1,1


Instead of analyzing individual logs, SOC analysts correlate events across systems to identify suspicious timelines.

# 5. Generate incident summaries

In [20]:
user = "alice"

timeline = pd.concat([
    auth[auth["user"] == user][["timestamp"]],
    endpoint[endpoint["user"] == user][["timestamp"]],
])

timeline.sort_values("timestamp")

,timestamp
0,2026-03-08 09:01:00
1,2026-03-08 09:02:00
2,2026-03-08 09:03:00
3,2026-03-08 09:04:00
0,2026-03-08 09:06:00
1,2026-03-08 09:07:00
2,2026-03-08 09:08:00


Creating a timeline helps analysts understand the sequence of events leading to a potential compromise.

# 6. Generate an incident summary (GenAI concept)

In [22]:
{
    "incident_id": "INC-TCK-001",
    "user": "alice",
    "severity": "high",
    "reasons": [
        "Suspicious email wording",
        "Dangerous attachment type",
        "Successful login after multiple failures",
        "Suspicious script/command execution"
    ],
    "timeline": [
        {"timestamp": "2026-03-08 08:58:00", "source": "email", "event": "Reported email: Urgent invoice overdue"},
        {"timestamp": "2026-03-08 09:04:00", "source": "auth", "event": "login / success from 185.220.101.4 (RU)"},
        {"timestamp": "2026-03-08 09:07:00", "source": "endpoint", "event": "powershell.exe executed"}
    ]
}

{'incident_id': 'INC-TCK-001',
 'user': 'alice',
 'severity': 'high',
 'reasons': ['Suspicious email wording',
  'Dangerous attachment type',
  'Successful login after multiple failures',
  'Suspicious script/command execution'],
 'timeline': [{'timestamp': '2026-03-08 08:58:00',
   'source': 'email',
   'event': 'Reported email: Urgent invoice overdue'},
  {'timestamp': '2026-03-08 09:04:00',
   'source': 'auth',
   'event': 'login / success from 185.220.101.4 (RU)'},
  {'timestamp': '2026-03-08 09:07:00',
   'source': 'endpoint',
   'event': 'powershell.exe executed'}]}

In [ ]:
def create_incidents(auth, email, endpoint):
    incidents = []

    for _, row in email.iterrows():

        user = row["user"]
        start = row["timestamp"]
        end = start + timedelta(minutes=30)

        # événements auth liés
        auth_events = auth[
            (auth["user"] == user) &
            (auth["timestamp"] >= start) &
            (auth["timestamp"] <= end)
        ]

        # événements endpoint liés
        endpoint_events = endpoint[
            (endpoint["user"] == user) &
            (endpoint["timestamp"] >= start) &
            (endpoint["timestamp"] <= end)
        ]

        # timeline
        timeline = []

        timeline.append({
            "timestamp": start,
            "source": "email",
            "event": f"Reported email: {row['subject']}"
        })

        for _, a in auth_events.iterrows():
            timeline.append({
                "timestamp": a["timestamp"],
                "source": "auth",
                "event": f"login {a['status']} from {a['ip']} ({a['country']})"
            })

        for _, e in endpoint_events.iterrows():
            timeline.append({
                "timestamp": e["timestamp"],
                "source": "endpoint",
                "event": f"{e['process_name']} executed"
            })

        timeline = sorted(timeline, key=lambda x: x["timestamp"])

        incident = {
            "user": user,
            "email_subject": row["subject"],
            "severity": "medium",
            "score": len(timeline),
            "reasons": [
                "Suspicious email reported",
                "Authentication activity after email",
                "Endpoint activity detected"
            ],
            "timeline": timeline
        }

        incidents.append(incident)

    return incidents

In [24]:
incidents = create_incidents(auth, email, endpoint)
incidents

[{'user': 'alice',
  'email_subject': 'Urgent invoice overdue',
  'severity': 'medium',
  'score': 8,
  'reasons': ['Suspicious email reported',
   'Authentication activity after email',
   'Endpoint activity detected'],
  'timeline': [{'timestamp': Timestamp('2026-03-08 08:58:00'),
    'source': 'email',
    'event': 'Reported email: Urgent invoice overdue'},
   {'timestamp': Timestamp('2026-03-08 09:01:00'),
    'source': 'auth',
    'event': 'login failed from 10.0.0.12 (FR)'},
   {'timestamp': Timestamp('2026-03-08 09:02:00'),
    'source': 'auth',
    'event': 'login failed from 10.0.0.12 (FR)'},
   {'timestamp': Timestamp('2026-03-08 09:03:00'),
    'source': 'auth',
    'event': 'login failed from 10.0.0.12 (FR)'},
   {'timestamp': Timestamp('2026-03-08 09:04:00'),
    'source': 'auth',
    'event': 'login success from 185.220.101.4 (RU)'},
   {'timestamp': Timestamp('2026-03-08 09:06:00'),
    'source': 'endpoint',
    'event': 'WINWORD.EXE executed'},
   {'timestamp': Timestam

In [28]:
print(generate_incident_summary(incidents))

Threat category:
Suspicious activity

Summary:
Fallback summary generated because incident format was invalid.

Why suspicious:
- Incident data could not be parsed correctly

Recommended actions:
1. Review the raw incident data
2. Validate the input format
3. Re-run the analysis

Analyst warning:
AI-generated support only. Human validation is required.

[Fallback used: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}]


In [26]:
print(generate_incident_summary("alice"))

Threat category:
Suspicious activity

Summary:
Incident involving alice.
Correlated events suggest potentially suspicious behavior.
This case should be reviewed by a human analyst.

Why suspicious:
- Limited structured data available
- Fallback mode used because LLM request failed

Recommended actions:
1. Review the related logs manually
2. Check authentication and endpoint timeline
3. Escalate if compromise is confirmed

Analyst warning:
AI-generated support only. Human validation is required.

[Fallback used: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}]


In [ ]:
print(generate_incident_summary("charlie"))


Potential phishing-related incident detected.

User: charlie

Signals observed:
- Suspicious email reported
- Multiple failed login attempts
- Endpoint process execution

Recommended actions:
1. Verify email sender and attachment
2. Reset user credentials
3. Investigate endpoint activity



In a production system, a GenAI model could generate structured summaries to help analysts understand incidents faster.

Limitations and Risks

GenAI can assist analysts but must not replace human judgment.

Potential risks include:
- hallucinations
- prompt injection through attacker-controlled log content
- exposure of sensitive log data

Therefore AI output should always be validated by a human analyst.

In [ ]:
# QWEN2.5:7b - The best in term of reasonning

start = time.time()

print(generate_incident_summary_local(incidents, model="qwen2.5:7b"))

end = time.time()

print(f"Times: {end - start:2f} seconds")

Threat category:
Phishing and Lateral Movement

Summary:
The incident involves multiple users reporting suspicious emails, followed by failed login attempts and endpoint activities such as the execution of various applications. The primary user, Alice, reported an "Urgent invoice overdue" email, which led to multiple failed login attempts from a French IP address (10.0.0.12) and successful login from another French IP (185.220.101.4). Bob and Charlie also reported suspicious emails, with Charlie's account showing a successful login from a French IP (10.0.0.23) and the execution of a browser application.

Why suspicious:
- Multiple users reported suspicious emails, indicating a potential phishing campaign.
- Failed login attempts from a specific IP address, suggesting an attacker probing for vulnerabilities.
- Successful login from a different IP address, indicating a successful breach.
- Execution of various applications on the endpoint, which could be used for further reconnaissance o

In [40]:
# MISTRAL - the fastest

start = time.time()

print(generate_incident_summary_local(incidents, model="mistral:latest"))

end = time.time()

print(f"Times: {end - start:2f} seconds")


Threat category:
Phishing and Potential Account Compromise

Summary:
Three users (Alice, Bob, and Charlie) received suspicious emails with subjects "Urgent invoice overdue", "Policy update", and "Reset your password". Each email was reported as such, and authentication and endpoint activities were detected after receiving the emails. Multiple failed login attempts and successful login from foreign IP addresses were observed, along with suspicious executables being run on endpoints.

Why suspicious:
1. The emails have subjects that are often used in phishing attempts.
2. The authentication and endpoint activities occurred shortly after receiving the emails.
3. The failed login attempts and successful login from foreign IP addresses are unusual and may indicate an attempt to gain unauthorized access.
4. The execution of suspicious executables (WINWORD.EXE, powershell.exe, cmd.exe, AcroRd32.exe, and chrome.exe) on endpoints is a potential indicator of malicious activity.

Recommended acti

In [41]:
# LLAMA3.2 - the lightest

start = time.time()

print((generate_incident_summary_local(incidents, model="llama3.2:latest")))

end = time.time()

print(f"Times: {end - start:2f} seconds")

Threat category:
Phishing/Drive-by Download

Summary:
Multiple users reported suspicious emails with urgent invoice overdues, which triggered authentication activity and endpoint activity. The incident includes login failures and subsequent login successes from unknown IP addresses.

Why suspicious:
- The emails were reported as suspicious, which may indicate phishing activity.
- The authentication activity and endpoint activity suggest that the users may have interacted with malicious content.
- The login failures and subsequent login successes from unknown IP addresses may indicate a drive-by download or a malicious script execution.

Recommended actions:
1. Investigate the IP addresses associated with the login successes to determine if they are legitimate or malicious.
2. Analyze the email content and attachments to determine if they contain any malicious links or files.
3. Review the endpoint activity to determine if any malicious software was executed.

Analyst warning:
Human val

## Constrained Reasoning - Less Hallucination + Scoring